In [1]:
import pandas as pd
import json

# 1. Load Data
df_visits = pd.read_csv('visits_restaurant_geolocated.csv')
df_biz = pd.read_csv('business_10k_restaurants.csv')

# 2. Merge to get restaurant names for every visit
# We join 'visits' with 'business' so we know that business_id "X" is actually "Tuna Bar"
df_merged = df_visits.merge(df_biz[['business_id', 'name', 'categories', 'stars']], on='business_id', how='left')

# 3. Aggregate history per User
# We create a natural language string describing their style.
def build_user_profile(group):
    # Take the last 10 visits (or all if <10)
    recent_visits = group.head(10)
    
    visit_descriptions = []
    for _, row in recent_visits.iterrows():
        # Handle cases where restaurant data might be missing (NaN)
        name = row['name'] if pd.notna(row['name']) else "Unknown Place"
        cats = row['categories'] if pd.notna(row['categories']) else "Food"
        visit_descriptions.append(f"- Visited {name} ({cats})")
    
    # Combine into one block of text
    return "User History:\n" + "\n".join(visit_descriptions)

# Apply to all users
user_profiles = df_merged.groupby('user_id').apply(build_user_profile).to_dict()

# 4. Save as JSON for fast lookup in your backend
with open('user_profiles.json', 'w') as f:
    json.dump(user_profiles, f)

print("Saved user_profiles.json! Example entry:")
# Print the first key-value pair to verify
first_user = list(user_profiles.keys())[0]
print(f"User: {first_user}")
print(user_profiles[first_user])

Saved user_profiles.json! Example entry:
User: ---zemaUC8WeJeWKqS6p9Q
User History:
- Visited Luckys Steakhouse (Restaurants, Seafood, Nightlife, Steakhouses, Bars, American (Traditional), Cocktail Bars, Wine Bars)


C:\Users\lebro\AppData\Local\Temp\ipykernel_18528\3070879640.py:29: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  user_profiles = df_merged.groupby('user_id').apply(build_user_profile).to_dict()


In [ ]:
import pandas as pd

# 1. Load the raw data
# Make sure these filenames match exactly what you have in your folder
df_business = pd.read_csv('business_10k_restaurants.csv')
df_visits = pd.read_csv('visits_restaurant_geolocated.csv')

# 2. Calculate "Popularity" 
# We count how many times each restaurant appears in the visits file
visit_counts = df_visits['business_id'].value_counts().reset_index()
visit_counts.columns = ['business_id', 'visit_count']

# 3. Merge Business Details with Popularity
# We use 'left' merge so we keep all restaurants, even if they have 0 visits
df_rag = pd.merge(df_business, visit_counts, on='business_id', how='left')

# Fill NaN (missing) visit counts with 0
df_rag['visit_count'] = df_rag['visit_count'].fillna(0).astype(int)

# 4. Create the 'rag_text' column
# This is the SINGLE string the AI will read to understand the restaurant.
df_rag['rag_text'] = (
    "Name: " + df_rag['name'].astype(str) + ". " +
    "Categories: " + df_rag['categories'].astype(str) + ". " +
    "City: " + df_rag['city'].astype(str) + ", " + df_rag['state'].astype(str) + ". " +
    "Rating: " + df_rag['stars'].astype(str) + " stars. " +
    "Review Count: " + df_rag['review_count'].astype(str) + ". " +
    "Popularity: " + df_rag['visit_count'].astype(str) + " visits in dataset."
)

# 5. Clean up and Save
# We only need the ID (for linking) and the Text (for the AI)
output_df = df_rag[['business_id', 'rag_text']]

output_df.to_csv('restaurant_rag_data.csv', index=False)

print(f"Successfully created 'restaurant_rag_data.csv' with {len(output_df)} restaurants.")
print("Sample content:")
print(output_df.head(1)['rag_text'].values[0])